🧹 Notebook 01: Data Cleaning & Feature Engineering

Questo notebook si occupa di:

1. Caricare il dataset grezzo (Sample-Superstore.csv).

2. Ispezionare la struttura, i tipi di dati e la presenza di valori nulli.

3. Correggere le tipologie di dato (conversione date, gestione codici postali).

4. Creare nuove colonne utili alle analisi successive (Feature Engineering).

5. Esportare il dataset pulito in data/superstore_cleaned.csv.

### 1. Importazione Librerie e Caricamento Dati

In [50]:
import pandas as pd
import numpy as np

# Definizione dei percorsi file
RAW_DATA_PATH = '../data/Sample-Superstore.csv'
CLEANED_DATA_PATH = '../data/superstore_cleaned.csv'

# Caricamento del dataset
# Nota: Usiamo encoding 'windows-1252' o 'latin1' se necessario per caratteri speciali
df = pd.read_csv(RAW_DATA_PATH, encoding="utf-8")

print(f"Dataset caricato con successo. Dimensione: {df.shape[0]} righe x {df.shape[1]} colonne.")

Dataset caricato con successo. Dimensione: 9800 righe x 18 colonne.


### 2. Ispezione Preliminare

In [51]:
# Mostra le prime 5 righe
display(df.head())

# Informazioni su tipi di colonne e valori non-nulli
df.info()

# Controllo diretto dei valori mancanti per colonna
print("\n--- Valori Mancanti per Colonna ---")
print(df.isnull().sum())

,Row ID,Order ID,Order Date,Ship Date,Ship Mode,Customer ID,Customer Name,Segment,Country,City,State,Postal Code,Region,Product ID,Category,Sub-Category,Product Name,Sales
0,1,CA-2017-152156,08/11/2017,11/11/2017,Second Class,CG-12520,Claire Gute,Consumer,United States,Henderson,Kentucky,42420.0,South,FUR-BO-10001798,Furniture,Bookcases,Bush Somerset Collection Bookcase,261.9600
1,2,CA-2017-152156,08/11/2017,11/11/2017,Second Class,CG-12520,Claire Gute,Consumer,United States,Henderson,Kentucky,42420.0,South,FUR-CH-10000454,Furniture,Chairs,"Hon Deluxe Fabric Upholstered Stacking Chairs,...",731.9400
2,3,CA-2017-138688,12/06/2017,16/06/2017,Second Class,DV-13045,Darrin Van Huff,Corporate,United States,Los Angeles,California,90036.0,West,OFF-LA-10000240,Office Supplies,Labels,Self-Adhesive Address Labels for Typewriters b...,14.6200
3,4,US-2016-108966,11/10/2016,18/10/2016,Standard Class,SO-20335,Sean O'Donnell,Consumer,United States,Fort Lauderdale,Florida,33311.0,South,FUR-TA-10000577,Furniture,Tables,Bretford CR4500 Series Slim Rectangular Table,957.5775
4,5,US-2016-108966,11/10/2016,18/10/2016,Standard Class,SO-20335,Sean O'Donnell,Consumer,United States,Fort Lauderdale,Florida,33311.0,South,OFF-ST-10000760,Office Supplies,Storage,Eldon Fold 'N Roll Cart System,22.3680


<class 'pandas.DataFrame'>
RangeIndex: 9800 entries, 0 to 9799
Data columns (total 18 columns):
 #   Column         Non-Null Count  Dtype  
---  ------         --------------  -----  
 0   Row ID         9800 non-null   int64  
 1   Order ID       9800 non-null   str    
 2   Order Date     9800 non-null   str    
 3   Ship Date      9800 non-null   str    
 4   Ship Mode      9800 non-null   str    
 5   Customer ID    9800 non-null   str    
 6   Customer Name  9800 non-null   str    
 7   Segment        9800 non-null   str    
 8   Country        9800 non-null   str    
 9   City           9800 non-null   str    
 10  State          9800 non-null   str    
 11  Postal Code    9789 non-null   float64
 12  Region         9800 non-null   str    
 13  Product ID     9800 non-null   str    
 14  Category       9800 non-null   str    
 15  Sub-Category   9800 non-null   str    
 16  Product Name   9800 non-null   str    
 17  Sales          9800 non-null   float64
dtypes: float64(2), int6

### 3. Pulizia e Conversione Tipi di Dato

In [52]:
# A. Conversione delle colonne data da stringa (str) a datetime
df['Order Date'] = pd.to_datetime(df['Order Date'], format='%d/%m/%Y')
df['Ship Date'] = pd.to_datetime(df['Ship Date'], format='%d/%m/%Y')

# B. Gestione dei valori mancanti in Postal Code
print(df[df['Postal Code'].isna()])
# I Postal Code mancanti appartengono alla città di Burlington, Vermont.
# Imputiamo con il codice postale noto o gestiamo il dato mantenendolo coerente.

df['Postal Code'] = df['Postal Code'].fillna(5401).astype(int).astype(str).str.zfill(5)

# C. Rimozione di eventuali righe completamente duplicate (se presenti)
initial_rows = len(df)
df = df.drop_duplicates()
removed_rows = initial_rows - len(df)
print(f"Righe duplicate rimosse: {removed_rows}")


      Row ID        Order ID Order Date  Ship Date       Ship Mode  \
2234    2235  CA-2018-104066 2018-12-05 2018-12-10  Standard Class   
5274    5275  CA-2016-162887 2016-11-07 2016-11-09    Second Class   
8798    8799  US-2017-150140 2017-04-06 2017-04-10  Standard Class   
9146    9147  US-2017-165505 2017-01-23 2017-01-27  Standard Class   
9147    9148  US-2017-165505 2017-01-23 2017-01-27  Standard Class   
9148    9149  US-2017-165505 2017-01-23 2017-01-27  Standard Class   
9386    9387  US-2018-127292 2018-01-19 2018-01-23  Standard Class   
9387    9388  US-2018-127292 2018-01-19 2018-01-23  Standard Class   
9388    9389  US-2018-127292 2018-01-19 2018-01-23  Standard Class   
9389    9390  US-2018-127292 2018-01-19 2018-01-23  Standard Class   
9741    9742  CA-2016-117086 2016-11-08 2016-11-12  Standard Class   

     Customer ID     Customer Name      Segment        Country        City  \
2234    QJ-19255      Quincy Jones    Corporate  United States  Burlington   
527

### 4. Feature Engineering (Creazione Nuove Variabili)

In [53]:
# A. Tempo di gestione e spedizione dell'ordine (in giorni)
df['Shipping Days'] = (df['Ship Date'] - df['Order Date']).dt.days

# B. Estrazione componenti temporali da Order Date
df['Order Year'] = df['Order Date'].dt.year
df['Order Month'] = df['Order Date'].dt.month
df['Order Month_Name'] = df['Order Date'].dt.strftime('%B')
df['Order YearMonth'] = df['Order Date'].dt.to_period('M')
df['Order DayOfWeek'] = df['Order Date'].dt.day_name()

# C. Calcolo delle Vendite Unitarie Approssimate
df['Unit Price Estimate'] = (df['Sales'] / df['Quantity']).round(2) if 'Quantity' in df.columns else df['Sales']

# Verifica delle nuove colonne create
print("\n--- Informazioni dopo il Feature Engineering ---")
df.info()


--- Informazioni dopo il Feature Engineering ---
<class 'pandas.DataFrame'>
RangeIndex: 9800 entries, 0 to 9799
Data columns (total 25 columns):
 #   Column               Non-Null Count  Dtype         
---  ------               --------------  -----         
 0   Row ID               9800 non-null   int64         
 1   Order ID             9800 non-null   str           
 2   Order Date           9800 non-null   datetime64[us]
 3   Ship Date            9800 non-null   datetime64[us]
 4   Ship Mode            9800 non-null   str           
 5   Customer ID          9800 non-null   str           
 6   Customer Name        9800 non-null   str           
 7   Segment              9800 non-null   str           
 8   Country              9800 non-null   str           
 9   City                 9800 non-null   str           
 10  State                9800 non-null   str           
 11  Postal Code          9800 non-null   str           
 12  Region               9800 non-null   str           

### 5. Salvataggio del Dataset Pulito

In [54]:
# Esportazione in formato CSV nella cartella data/
df.to_csv(CLEANED_DATA_PATH, index=False)
print(f"Dataset pulito ed elaborato salvato con successo in: {CLEANED_DATA_PATH}")

Dataset pulito ed elaborato salvato con successo in: ../data/superstore_cleaned.csv
